# Fact Orders

### Data Reading

In [0]:
df = spark.sql("SELECT * FROM DEV.SILVER.orders_silver")
df.dispaly()

In [0]:
df_dimCust = spark.sql("select DimCustomerKey, customer_id as dim_customer_id from DEV.GOLD.DimCustomers")

df_dimproduct =  spark.sql("select product_id as DimProduct_id from DEV.GOLD.DimProducts")

### Fact Dataframe

In [0]:
df_fact = df.join(df_dimCust, df["customer_id"] == df_dimCust["dim_customer_id"], how = 'left').join(df_dimproduct, df["product_id"] == df_dimproduct       ["DimProduct_id"], how='left')

df_fact_new = df_fact.drop('dim_customer_id','DimProduct_id','customer_id','product_id')

df_fact_new.display()

### Upsert on Fact Tables

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists("DEV.GOLD.FactOrders"):
    dlt_obj =DeltaTable.forName(spark, "Dev.Gold.FactOrders")

    dlt_obj.alias("trg").merge(df_fact_new.alias("src"), "trg.order_id == src.order_id AND trg.DimCustomerKey = src.DimCustomerKey AND trg.DimProductsKey = src.DimProductKey")\
        .whenNotMatchedUpdateAll()\
        t.whenMatchedInserAll()

else:
    df_fact_new.write.format("delta").option("abfss://gold@adbstoragepro.dfs.core.windows.net/FactOrders").saveAsTable("Dev.Gold.FactOrders")


In [0]:
%sql
SELECT * FROM Dev.Gold.FactOrders;